In [1]:
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.distributions as dist

In [2]:
N_MIXTURE = 5
RADIUS = 5.0
MAJOR_VAR, MINOR_VAR = 1.5, 0.1
ANGLE = 2 * math.pi / N_MIXTURE


def rotation_matrix(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s], [s, c]])


def _component_mean(i):
    return RADIUS * np.array([np.cos(i * ANGLE), np.sin(i * ANGLE)])


def _component_cov(i):
    R_i = rotation_matrix(i * ANGLE)
    C0 = np.diag([MAJOR_VAR, MINOR_VAR])
    return R_i @ C0 @ R_i.T


weights = np.ones(N_MIXTURE) / N_MIXTURE
mus = np.stack([_component_mean(i) for i in range(N_MIXTURE)])
Cs = np.stack([_component_cov(i) for i in range(N_MIXTURE)])

target = dist.MixtureSameFamily(
    dist.Categorical(torch.tensor(weights)),
    dist.MultivariateNormal(torch.tensor(mus), torch.tensor(Cs)),
)

In [3]:
def simulate_dynamics(x0, lam=2.0, dt=0.01, n_steps=10**6, seed=None):
    """Euler-Maruyama for the per-component Langevin SDE plus a Poisson rotation clock."""
    rng = np.random.default_rng(seed)
    R_rot = rotation_matrix(ANGLE)
    d = x0.shape[0]
    traj = np.zeros((n_steps + 1, d))
    traj[0] = x0
    k = 0
    for t in range(n_steps):
        drift = -np.linalg.solve(Cs[k], traj[t] - mus[k])
        proposal = traj[t] + dt * drift + np.sqrt(dt) * rng.standard_normal(d)
        if rng.random() < lam * dt:
            traj[t + 1] = R_rot @ proposal
            k = (k + 1) % N_MIXTURE
        else:
            traj[t + 1] = proposal
    return traj

In [4]:
# removed conditional simulation 
res = simulate_dynamics(mus[0], lam=2.0, dt=0.01, n_steps=10**6, seed=0)

In [5]:
torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SUBSAMPLE = 50
LAG = 1

data = torch.tensor(res[10**4::SUBSAMPLE], dtype=torch.float)
N, d = data.shape
n_train = int(0.01 * N) # modify to be a lot smaller 

train_now = data[:n_train]
train_future = data[LAG:n_train + LAG]
test_now = data[n_train:-LAG]
test_future = data[n_train + LAG:]

print(f'[dataset] size of dataset {N}*{d}')
print(train_now.mean(), train_now.max(), train_now.min(),
      train_future.pow(2).mean().sqrt())

[dataset] size of dataset 19801*2
tensor(-0.0989) tensor(7.2572) tensor(-6.4978) tensor(3.7458)


In [6]:
class Velocity(nn.Module):
    def __init__(self, state_dim=2, cond_dim=2, hidden=500, depth=6):
        super().__init__()
        widths = [state_dim + cond_dim + 1] + [hidden] * depth + [state_dim]
        layers = []
        for i in range(len(widths) - 1):
            layers.append(nn.Linear(widths[i], widths[i + 1]))
            if i < len(widths) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
        n_params = sum(p.numel() for p in self.parameters())
        print(f'[Velocity] num params: {n_params:,}')

    def forward(self, zt, t, cond):
        return self.net(torch.cat([zt, cond, t[:, None]], dim=1))

In [7]:
class Interpolants:
    def __init__(self, sigma_coef=1.0):
        self.sigma_coef = sigma_coef
        print(f'[Interpolants] sigma_coef = {sigma_coef}')

    # coefficient schedules (all return shape matching t)
    def alpha(self, t):     return 1 - t
    def alpha_dot(self, t): return -torch.ones_like(t)
    def beta(self, t):      return t ** 2
    def beta_dot(self, t):  return 2 * t
    def sigma(self, t):     return self.sigma_coef * (1 - t)
    def sigma_dot(self, t): return -self.sigma_coef * torch.ones_like(t)
    def gamma(self, t):     return self.sigma(t) * t.sqrt()

    def interpolate(self, z0, z1, noise, t):
        t_ = t[:, None]
        return self.alpha(t_) * z0 + self.beta(t_) * z1 + self.gamma(t_) * noise

    def drift_target(self, z0, z1, noise, t):
        t_ = t[:, None]
        return (self.alpha_dot(t_) * z0
                + self.beta_dot(t_) * z1
                + (self.sigma_dot(t_) * t_.sqrt()) * noise)

In [8]:
class EulerMaruyama:
    """Generative SDE solver: x0 -> x1 via dz = b(z, t, cond) dt + sigma(t) dW."""
    def __init__(self, interpolant, t_min=0.0, t_max=1.0):
        self.I = interpolant
        self.t_min, self.t_max = t_min, t_max

    @torch.no_grad()
    def sample(self, model, x0, n_steps=2000):
        cond = x0
        zt = x0.clone()
        ts = torch.linspace(self.t_min, self.t_max, n_steps, device=x0.device)
        dt = ts[1] - ts[0]
        ones = torch.ones(zt.shape[0], device=zt.device)
        for t_scalar in ts:
            t = t_scalar * ones
            b = model(zt, t, cond=cond)
            g = self.I.sigma(t)[:, None]
            zt_mean = zt + b * dt
            zt = zt_mean + g * torch.randn_like(zt_mean) * dt.sqrt()
        return zt_mean

In [9]:
class Trainer:
    def __init__(self, model, interpolant, sampler, *,
                 train_now, train_future, test_now, test_future,
                 batch_size=10_000, base_lr=1e-3, max_steps=3000,
                 sample_every=2000, print_every=10,
                 max_grad_norm=1.0, cosine_lr=True, device='cuda'):
        self.model = model.to(device)
        self.I = interpolant
        self.sampler = sampler
        self.device = device
        self.base_lr = base_lr
        self.max_steps = max_steps
        self.sample_every = sample_every
        self.print_every = print_every
        self.max_grad_norm = max_grad_norm
        self.cosine_lr = cosine_lr
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=base_lr)
        self.t_dist = torch.distributions.Uniform(low=0.0, high=1.0)
        self.step = 0
        self.epoch = 0
        self.train_loader = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(train_now, train_future),
            batch_size=batch_size, shuffle=True)
        self.test_loader = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(test_now, test_future),
            batch_size=batch_size, shuffle=False)
        self.ref_batch = self._make_reference(n_train=100, n_test=100)
        print(f'[Optimizer] AdamW, lr={base_lr}')

    def _make_reference(self, n_train, n_test):
        nt, ft = next(iter(self.train_loader))
        ne, fe = next(iter(self.test_loader))
        ref_now = torch.cat([nt[:n_train], ne[:n_test]]).to(self.device)
        ref_future = torch.cat([ft[:n_train], fe[:n_test]]).to(self.device)
        return ref_now, ref_future

    def _make_batch(self, now, future):
        t = self.t_dist.sample((future.shape[0],)).to(self.device)
        noise = torch.randn_like(future)
        zt = self.I.interpolate(z0=now, z1=future, noise=noise, t=t)
        target = self.I.drift_target(z0=now, z1=future, noise=noise, t=t)
        return {'zt': zt, 't': t, 'cond': now, 'target': target}

    def _loss(self, D):
        out = self.model(D['zt'], D['t'], cond=D['cond'])
        return (out - D['target']).pow(2).sum(-1).mean()

    def _optimizer_step(self):
        grad_norm = torch.nn.utils.clip_grad_norm_(
            self.model.parameters(), max_norm=self.max_grad_norm)
        self.opt.step()
        self.opt.zero_grad(set_to_none=True)
        self.step += 1
        return grad_norm

    def _adjust_lr(self):
        if not self.cosine_lr:
            return
        scale = self.step / self.max_steps
        lr = self.base_lr * 0.5 * (1 + math.cos(math.pi * scale))
        for g in self.opt.param_groups:
            g['lr'] = lr
        # print(f'[Scheduler] lr = {lr:.3e}')

    @torch.no_grad()
    def _eval_reference(self):
        self.model.eval()
        now, future = self.ref_batch
        zT = self.sampler.sample(self.model, now)
        err = (zT - future).norm(dim=1) / future.norm(dim=1).mean()
        n = err.shape[0] // 2
        return err[:n].mean().item(), err[n:].mean().item()

    def fit(self):
        t0 = time.time()
        print('[Training] start')
        while self.step < self.max_steps:
            for now, future in self.train_loader:
                if self.step >= self.max_steps:
                    return
                now, future = now.to(self.device), future.to(self.device)
                self.model.train()
                loss = self._loss(self._make_batch(now, future))
                loss.backward()
                grad_norm = self._optimizer_step()
                # if self.step % self.print_every == 0:
                #     mins = (time.time() - t0) / 60
                #     print(f'step {self.step:5d}  loss {loss.item():.4f}  '
                #           f'grad {grad_norm:.2f}  {mins:.2f} min')
                if self.step % self.sample_every == 0:
                    tr, te = self._eval_reference()
                    # print(f'   ref relative err -- train {tr:.3f}  test {te:.3f}')
            self.epoch += 1
            self._adjust_lr()

In [10]:
interpolant = Interpolants(sigma_coef=1.0)
sampler = EulerMaruyama(interpolant)
model = Velocity(state_dim=2, cond_dim=2, hidden=500, depth=6)

trainer = Trainer(
    model, interpolant, sampler,
    train_now=train_now, train_future=train_future,
    test_now=test_now, test_future=test_future,
    batch_size=10_000, base_lr=1e-3, max_steps=3000,
    sample_every=2000, device=device,
)

# Uncomment to train from scratch (~minutes on a single GPU):
trainer.fit()


[Interpolants] sigma_coef = 1.0
[Velocity] num params: 1,256,502
[Optimizer] AdamW, lr=0.001
[Training] start


In [11]:
model.eval()

n_samples = 2000  # Smaller for quick demo
n_rollout = 4
em_steps = 1000 # Smaller for quick demo

x = torch.tensor(mus[0:1], dtype=torch.float32).repeat(n_samples, 1).to(device)

t0 = time.time()
with torch.no_grad():
    for k in range(n_rollout):
        x = sampler.sample(model, x, n_steps=em_steps)
        # print(f'[Sampler] rollout {k + 1}/{n_rollout}, '
        #       f'finished in {(time.time() - t0) / 60:.2f} minutes')

forecast = x.cpu()

In [12]:
# added for comparing the diffs.
print(f'Forecast mean: {forecast.mean(dim=0)}')
print(f'Forecast std: {forecast.std(dim=0)}')

Forecast mean: tensor([-0.4954, -0.3743])
Forecast std: tensor([3.7070, 3.6381])
